# Week 3.5 — Multilingual Presentation Figures, MBERT
**Project**: Quantum Kernels — Multilingual

**Why this notebook:** the Week 3 multilingual notebook focused on the novel
KTA-predicts-gap analysis and skipped the two standard per-dataset figures every
other dataset in the paper has — the **comparison bar chart** and the
**learning curve**. This notebook adds them per language, so the multilingual
section is presented consistently with SST-2, AG News, CIFAR-10, etc.

**Fast:** reuses the kernel matrices Week 3 already saved
(`multiling_mbert_<lang>_kernel_matrices.npz`). No kernel recomputation — the learning
curve slices the saved 1000x1000 matrix. Runtime: ~2 min for all languages.

**Inputs:** `multiling_mbert_<lang>_embeddings.npz`, `multiling_mbert_<lang>_kernel_matrices.npz`,
`multiling_mbert_<lang>_baseline_results.pkl` (all from Week 2 + Week 3)

In [ ]:
!pip install pennylane pennylane-lightning scikit-learn numpy --quiet
import numpy as np, json, pickle, os, shutil, warnings
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
warnings.filterwarnings('ignore')
print("Loaded.")

---
## Step 0: Storage

In [ ]:
TAG = 'multiling_mbert'
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")
def _persist_and_download(path):
    if os.path.abspath(SAVE_DIR) != os.getcwd():
        shutil.copy(path, os.path.join(SAVE_DIR, os.path.basename(path)))
        print(f"  saved -> {SAVE_DIR}/{os.path.basename(path)}")
    try:
        from google.colab import files
        files.download(path)
        print(f"  download -> {os.path.basename(path)}")
    except Exception as e:
        print(f"  (download skipped: {type(e).__name__} — saved above)")
def checkpoint(filename, save_fn):
    save_fn(filename); _persist_and_download(filename)
def save_figure(fig, name):
    for ext in ('png','pdf'):
        p = f"{name}.{ext}"; fig.savefig(p, bbox_inches='tight', dpi=150); _persist_and_download(p)
def _find(fname):
    dp = os.path.join(SAVE_DIR, fname); return dp if os.path.exists(dp) else fname
print("Helpers ready (Drive + download).")

---
## Step 1: Config

In [ ]:
LANGUAGES = ['english', 'spanish', 'german', 'arabic', 'chinese', 'hindi']
def mf1(yt, yp): return float(f1_score(yt, yp, average='macro'))
print("Config ready.")

---
## Step 2: Per-language comparison bars + learning curves

For each language with saved Week 3 results: rebuild the quantum-vs-classical
comparison bar chart and the learning curve (sliced from the saved kernel matrix),
matching the figure style used for every other dataset.

In [ ]:
train_sizes = [50, 100, 200, 400, 700, 1000]
all_lc = {}   # learning curves collected for the combined figure

for lang in LANGUAGES:
    ltag = f'{TAG}_{lang}'
    emb_path = _find(f'{ltag}_embeddings.npz')
    km_path  = _find(f'{ltag}_kernel_matrices.npz')
    bl_path  = _find(f'{ltag}_baseline_results.pkl')
    if not (os.path.exists(emb_path) and os.path.exists(km_path) and os.path.exists(bl_path)):
        print(f"[skip] {lang}: missing files (run Week 2 + Week 3 first)")
        continue

    print(f"\n=== {lang} ===")
    data = np.load(emb_path)
    X_tr = data['X_tr_subset'].astype(np.float64); y_tr = data['y_tr_subset']
    X_te = data['X_te_subset'].astype(np.float64); y_te = data['y_te_subset']
    km = np.load(km_path); K_train = km['K_train_quantum']; K_test = km['K_test_quantum']
    with open(bl_path, 'rb') as f: baselines = pickle.load(f)

    # quantum result
    yp = SVC(kernel='precomputed', C=1.0).fit(K_train, y_tr).predict(K_test)
    acc_q = float(accuracy_score(y_te, yp)); f1_q = mf1(y_te, yp)
    results_all = {n: {'accuracy': v['accuracy'], 'f1': v['f1']} for n, v in baselines.items()}
    results_all['Quantum (IQP)'] = {'accuracy': acc_q, 'f1': f1_q}

    # --- comparison bar chart (matches other datasets) ---
    names = list(results_all.keys())
    accs = [results_all[n]['accuracy'] for n in names]; f1s = [results_all[n]['f1'] for n in names]
    colors = ['#5B8DB8','#5B8DB8','#5B8DB8','#D95F02']
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    for ax, vals, ylab in [(axes[0], accs, 'Accuracy'), (axes[1], f1s, 'Macro F1')]:
        ax.bar(names, vals, color=colors, edgecolor='white', width=0.6)
        ax.set_ylim(0.2, 1.0); ax.set_title(f'{ylab} — {lang} (MBERT) N=1000'); ax.set_ylabel(ylab)
        for i, v in enumerate(vals): ax.text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=11)
        ax.set_xticklabels(names, rotation=15, ha='right')
    fig.legend(handles=[mpatches.Patch(color='#5B8DB8',label='Classical'), mpatches.Patch(color='#D95F02',label='Quantum IQP')],
               loc='upper center', ncol=2, bbox_to_anchor=(0.5,1.02))
    plt.suptitle(f'Quantum vs Classical — {lang} sentiment (MBERT, N=1000)', y=1.06)
    plt.tight_layout(); plt.show()
    save_figure(fig, f'{ltag}_fig1_comparison')

    # --- learning curve (sliced from saved matrix) ---
    lc = {'Quantum (IQP)': [], 'Linear': [], 'RBF': []}
    for n in train_sizes:
        Kn, Kn_test = K_train[:n,:n], K_test[:,:n]
        ypq = SVC(kernel='precomputed', C=1.0).fit(Kn, y_tr[:n]).predict(Kn_test)
        lc['Quantum (IQP)'].append(mf1(y_te, ypq))
        for cn, ck in [('Linear','linear'), ('RBF','rbf')]:
            clf = SVC(kernel=ck, C=1.0, gamma='scale' if ck=='rbf' else 1.0, random_state=42).fit(X_tr[:n], y_tr[:n])
            lc[cn].append(mf1(y_te, clf.predict(X_te)))
    all_lc[lang] = lc

    fig2 = plt.figure(figsize=(8, 4))
    for m, vals in lc.items():
        ls = '-' if m=='Quantum (IQP)' else '--'; lw = 2.5 if m=='Quantum (IQP)' else 1.8
        plt.plot(train_sizes, vals, label=m, color={'Quantum (IQP)':'#D95F02','Linear':'#5B8DB8','RBF':'#2D9C4A'}[m],
                 linestyle=ls, linewidth=lw, marker='o', markersize=5)
    plt.xlabel('N'); plt.ylabel('Macro F1'); plt.title(f'Learning curves — {lang} (MBERT)')
    plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
    save_figure(fig2, f'{ltag}_fig2_learning_curves')

    print(f"  quantum macro-F1={f1_q:.4f}  figures saved")

print(f"\nProcessed: {list(all_lc.keys())}")

---
## Step 3: Combined cross-language learning-curve panel

A single figure showing the quantum learning curve for all languages together —
useful as one compact paper figure instead of six separate ones.

In [ ]:
if all_lc:
    fig, ax = plt.subplots(figsize=(9, 5))
    cmap = plt.cm.tab10(np.linspace(0, 1, len(all_lc)))
    for (lang, lc), col in zip(all_lc.items(), cmap):
        ax.plot(train_sizes, lc['Quantum (IQP)'], marker='o', markersize=4,
                color=col, label=lang)
    ax.set_xlabel('Training set size (N)'); ax.set_ylabel('Quantum macro F1')
    ax.set_title(f'Quantum kernel learning curves across languages (MBERT)')
    ax.legend(title='Language', fontsize=9); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
    save_figure(fig, f'{TAG}_combined_learning_curves')

    # save the learning curve data
    checkpoint(f'{TAG}_learning_curves.json',
               lambda p: json.dump({l: lc for l, lc in all_lc.items()}, open(p,'w'), indent=2))
    print("Combined figure + data saved.")
else:
    print("No languages processed — run Week 2 + Week 3 first.")

---
## Week 3.5 done — MBERT

### Saved per language:
- `multiling_mbert_<lang>_fig1_comparison.png/.pdf` — quantum vs classical bars
- `multiling_mbert_<lang>_fig2_learning_curves.png/.pdf` — per-language learning curve
### And combined:
- `multiling_mbert_combined_learning_curves.png/.pdf` — all languages in one panel
- `multiling_mbert_learning_curves.json` — the data

### Now the multilingual section matches the rest of the paper:
Every other dataset has comparison bars + a learning curve, and now multilingual
does too — per language, plus the unique KTA-predicts-gap scatter from Week 3.
The multilingual results can be presented in the same format as everything else.